# Fast segment classification with TF-IDF

CPU-only word (1-2 gram) and character (3-5 gram) features with logistic regression; LinearSVC is a challenger. The direct cascade has one scope and three four-class skill heads. The hierarchical cascade shares scope and uses presence then band for each skill.

Run from the project root with the workspace `.venv` kernel. Features and heads learn from train only. Dev selects settings; test is a separate opt-in final report. Candidate annotation bundles remain separate. Out-of-scope `N` is not absence `0`.

Training respects canonical text-quality exclusions and a 10-character minimum. Evaluation retains all nonblank candidates, including short texts, so coverage is explicit. Models may retain sensitive vocabulary: save only to approved protected storage and load only trusted joblib artifacts. No provider or tracing calls are made.

In [9]:
from pathlib import Path
from dataclasses import asdict
from itertools import product
from time import perf_counter

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from reflection_assessment_feedback.experiment_config import load_experiment_config
from reflection_assessment_feedback.segment_classification_ml import (
    MLSettings, SegmentClassifierML, eligible_rows,
)
from reflection_assessment_feedback.data import TARGET_COLUMNS

PROJECT_ROOT = Path.cwd().resolve()
assert (PROJECT_ROOT / 'config' / 'experiment.toml').is_file(), 'Run from the prebi_v1 root.'
load_dotenv(PROJECT_ROOT / '.env', override=False)
EXPERIMENT_CONFIG = load_experiment_config()
SPLIT_DIR = PROJECT_ROOT / EXPERIMENT_CONFIG['paths']['provisional_split_directory']
MINIMUM_TEXT_CHARACTERS = 10
RUN_TRAINING = False
RUN_TEST_EVALUATION = False
SAVE_MODEL = False
INCLUDE_LINEAR_SVC = True
REGULARIZATION_VALUES = (0.3, 1.0, 3.0)
CLASS_WEIGHTS = (None, 'balanced')
SEED: int = int(EXPERIMENT_CONFIG['classifier']['seed'])

print('CPU-only TF-IDF workflow ready; training, test evaluation, and saving are opt-in.')

CPU-only TF-IDF workflow ready; training, test evaluation, and saving are opt-in.


## Canonical splits and text-quality audit

Verify candidate alignment with the scope files and author/document separation. Report counts only, never reflection text. The minimum-length policy is fixed before model selection; adjust it using train/dev evidence only. Canonical quality exclusions can be stricter than the local minimum.

In [2]:
split_rows = {}
audit = []
for split in ('train', 'dev', 'test'):
    rows = pd.read_csv(
        SPLIT_DIR / f'provisional_{split}_modeling_wide.csv',
        dtype={'candidate_id': str, 'document_id': str, 'author_id': str},
    )
    scope_rows = pd.read_csv(
        SPLIT_DIR / f'provisional_{split}_scope.csv', dtype={'candidate_id': str},
    )
    assert rows['candidate_id'].is_unique and scope_rows['candidate_id'].is_unique
    assert set(rows['candidate_id']) == set(scope_rows['candidate_id'])
    pd.testing.assert_series_equal(
        rows.set_index('candidate_id')['scope_target'].sort_index(),
        scope_rows.set_index('candidate_id')['scope_target'].sort_index(),
        check_dtype=False,
    )
    rows['text'] = rows['text'].fillna('').astype(str)
    assert rows['text'].str.strip().ne('').all(), 'Blank evaluation text needs explicit handling.'
    assert rows['split'].eq(split).all()
    assert rows.loc[rows['scope_target'].eq(1), list(TARGET_COLUMNS.values())].notna().all().all()
    split_rows[split] = rows
    eligible = eligible_rows(rows, MINIMUM_TEXT_CHARACTERS)
    audit.append({
        'split': split, 'candidate_rows': len(rows),
        'unique_segments': rows['segment_id'].nunique(),
        'eligible_training_policy': int(eligible.sum()),
        'excluded_training_policy': int((~eligible).sum()),
        'short_rows': int(rows['text'].str.strip().str.len().lt(MINIMUM_TEXT_CHARACTERS).sum()),
    })
for left, right in [('train', 'dev'), ('train', 'test'), ('dev', 'test')]:
    for column in ['document_id', 'author_id', 'segment_id']:
        assert not (set(split_rows[left][column]) & set(split_rows[right][column])), f'{column} leakage'

display(pd.DataFrame(audit).set_index('split'))
training_eligible = split_rows['train'].loc[eligible_rows(split_rows['train'], MINIMUM_TEXT_CHARACTERS)]
label_counts = {'scope': training_eligible['scope_target'].value_counts().reindex([0, 1, 2, 3], fill_value=0)}
for dimension, column in TARGET_COLUMNS.items():
    label_counts[dimension] = (
        training_eligible.loc[training_eligible['scope_target'].eq(1), column]
        .value_counts().reindex([0, 1, 2, 3], fill_value=0)
    )
display(pd.DataFrame(label_counts).rename_axis('class'))

,candidate_rows,unique_segments,eligible_training_policy,excluded_training_policy,short_rows
split,,,,,
train,7107,7065,7107,0,0
dev,1823,1811,1823,0,0
test,905,903,905,0,0


,scope,SW,UA,HA
class,,,,
0,342,3980,3837,5713
1,6765,2275,1976,697
2,0,489,623,317
3,0,21,329,38


## Train and compare on dev

Enable `RUN_TRAINING` in the setup cell. The default grid compares both cascades, logistic regression and LinearSVC, three regularization strengths, and unweighted/balanced heads. Selection uses mean end-to-end dev macro-F1 across SW/UA/HA over gold-supported labels. Reports also show fixed-label metrics and zero-support classes explicitly. Do not select settings from test results.

The vocabulary is shared across a cascade's heads, learned from eligible train text only. Skill heads use gold in-scope rows; band heads use only positive skill rows. A constant head handles a single observed class; a band head with no positive examples stops with an error rather than inventing a band.

In [8]:
results = []
best_model = None
best_dev_score = float('-inf')
if RUN_TRAINING:
    estimators = ('logistic', 'linear_svc') if INCLUDE_LINEAR_SVC else ('logistic',)
    for architecture, estimator, regularization, class_weight in product(
        ('direct', 'hierarchical'), estimators, REGULARIZATION_VALUES, CLASS_WEIGHTS,
    ):
        settings = MLSettings(
            architecture=architecture, estimator=estimator,
            regularization=regularization, class_weight=class_weight,
            minimum_text_characters=MINIMUM_TEXT_CHARACTERS, seed=SEED,
        )
        started = perf_counter()
        model = SegmentClassifierML(settings).fit(split_rows['train'])
        training_seconds = perf_counter() - started
        started = perf_counter()
        metrics = model.evaluate(split_rows['dev'])
        evaluation_seconds = perf_counter() - started
        results.append({
            **asdict(settings), 'dev_macro_f1': metrics['selection_macro_f1'],
            'training_seconds': training_seconds, 'dev_evaluation_seconds': evaluation_seconds,
        })
        if metrics['selection_macro_f1'] > best_dev_score:
            best_model = model
            best_dev_score = metrics['selection_macro_f1']
        print(f'{architecture}, {estimator}, C={regularization}, weight={class_weight}: '
              f'dev={metrics["selection_macro_f1"]:.3f}, train={training_seconds:.1f}s')
    assert best_model is not None, 'The comparison grid produced no model.'
    leaderboard = pd.DataFrame(results).sort_values('dev_macro_f1', ascending=False)
    display(leaderboard)
    print('Selected settings:', asdict(best_model.settings))
else:
    print('Training disabled. Set RUN_TRAINING=True in setup and rerun this cell.')

Training disabled. Set RUN_TRAINING=True in setup and rerun this cell.


In [4]:
def show_reports(metrics):
    summary_rows = []
    for task, report in metrics['reports'].items():
        table = pd.DataFrame({
            label: values for label, values in report.items() if isinstance(values, dict)
        }).T
        print(task)
        display(table)
        summary_rows.append({'task': task, 'fixed_label_macro_f1': report['macro avg']['f1-score']})
    display(pd.DataFrame(summary_rows))

if best_model is not None:
    print('Selected-model dev results; zero-support classes are not evidence of performance.')
    show_reports(best_model.evaluate(split_rows['dev']))
if RUN_TEST_EVALUATION:
    if best_model is None:
        raise RuntimeError('Train and select on dev before final test evaluation.')
    print('Final test results: do not use these to change settings.')
    test_metrics = best_model.evaluate(split_rows['test'])
    show_reports(test_metrics)
else:
    print('Final test evaluation disabled.')

Final test evaluation disabled.


## Save the selected model for G2

Enable `SAVE_MODEL` only after reviewing the dev comparison. Set `PREBI_DATA_ROOT` to approved storage outside the source tree. The model and comparison metadata are stored under `segment-classification-ml/`; filenames include the model hash and existing different artifacts are not overwritten. Models retain vocabulary and must remain private. Only load your own trusted joblib files.

In `04_generate_g2.ipynb`, set `PREDICTION_BACKEND='tfidf'` and `ML_MODEL_PATH` to the saved model path. Enable `WRITE_ML_PREDICTIONS` there to generate local, model-fingerprinted predictions before the existing G2 validation and generation steps. GBERT remains the default backend.

In [7]:
import hashlib
import json
import os
import tempfile
from reflection_assessment_feedback.experiment_config import experiment_config_sha256

if SAVE_MODEL:
    if best_model is None:
        raise RuntimeError('Train and select a model before saving.')
    root_value = os.getenv('PREBI_DATA_ROOT')
    if not root_value:
        raise RuntimeError('Set PREBI_DATA_ROOT to approved protected storage.')
    protected_root = Path(root_value).expanduser().resolve()
    if protected_root == PROJECT_ROOT or PROJECT_ROOT in protected_root.parents:
        raise ValueError('Protected storage must be outside the project source tree.')
    model_directory = protected_root / 'segment-classification-ml'
    model_directory.mkdir(parents=True, exist_ok=True)
    best_model.training_metadata['split_sha256'] = {
        split: hashlib.sha256((SPLIT_DIR / f'provisional_{split}_modeling_wide.csv').read_bytes()).hexdigest()
        for split in ('train', 'dev')
    }
    with tempfile.NamedTemporaryFile(dir=model_directory, suffix='.joblib', delete=False) as temporary:
        staged_model = Path(temporary.name)
    try:
        best_model.save(staged_model)
        model_sha256 = hashlib.sha256(staged_model.read_bytes()).hexdigest()
        model_path = model_directory / f'model-{model_sha256}.joblib'
        os.chmod(staged_model, 0o600)
        if not model_path.exists():
            os.replace(staged_model, model_path)
    finally:
        staged_model.unlink(missing_ok=True)
    metadata = {
        'experiment_config_sha256': experiment_config_sha256(),
        'model_sha256': model_sha256, 'settings': asdict(best_model.settings),
        'training': best_model.training_metadata, 'dev_comparison': results,
        'selected_dev_metrics': best_model.evaluate(split_rows['dev']),
    }
    metadata_path = model_path.with_suffix('.json')
    if metadata_path.exists():
        if json.loads(metadata_path.read_text(encoding='utf-8')) != metadata:
            raise ValueError('Refusing to overwrite different model metadata.')
    else:
        with metadata_path.open('x', encoding='utf-8') as output:
            os.chmod(metadata_path, 0o600)
            json.dump(metadata, output, indent=2)
    print('Saved trusted model:', model_path)
else:
    print('Model saving disabled.')

Model saving disabled.
